# 03 DESeq2 で2つの条件を比較する

shControl と shPPP3CA で、遺伝子ごとに RNA 量を比べます。RNA-seq のカウントの比較によく使われる DESeq2 という方法を、Python 版の PyDESeq2 で実行します。

DESeq2 は、次の3つを順に行います。

1. 試料ごとの読み取った量の違いを調整する係数（サイズファクター）を求める
2. 遺伝子ごとに、同じ条件の試料のあいだのばらつきを見積もる
3. 条件の違いによる変化の大きさ（log₂ fold change）と、その確からしさ（p値）を求める

## 準備

下のセルを実行して、配布元から `samples.csv` をダウンロードします。

In [1]:
%pip install -q pydeseq2==0.5.4 anndata==0.12.10
import warnings
warnings.filterwarnings("ignore")  # 計算には影響しない警告を表示しない

DATA_URL = "https://pub-fcc6b90fb32448a4b3b455e664df76f0.r2.dev/e2f1-rnaseq/v1/"
!wget -q -nc {DATA_URL}samples.csv
!ls -l samples.csv

-rw-r--r-- 1 root root 213 Oct 25 05:54 samples.csv


## 前のノートブックの結果を読み込む

02_QC_PCA の最後でダウンロードした `results_02.zip` を使います。下のセルを実行し、「ファイルを選択」を押して `results_02.zip` を選んでください。zip の中の `counts_filtered.csv`, `vst.csv` を、このノートブックで使います。

(見つからない場合→ https://pub-fcc6b90fb32448a4b3b455e664df76f0.r2.dev/e2f1-rnaseq/v1/results_02.zip)

In [2]:
import zipfile
import os
from pathlib import Path

PREV_ZIP = "results_02.zip"
NEED = ['counts_filtered.csv', 'vst.csv']

if not all(Path(n).exists() for n in NEED):
    if Path(PREV_ZIP).exists():
        zipfile.ZipFile(PREV_ZIP).extractall()
    else:
        from google.colab import files
        for name in files.upload():
            if name.endswith(".zip"):
                zipfile.ZipFile(name).extractall()

missing = [n for n in NEED if not Path(n).exists()]
if missing:
    print("見つからないファイルがあります:", missing)
    print("このセルをもう一度実行して zip を選び直してください")
else:
    print("読み込みました:", NEED)

Saving results_02.zip to results_02.zip
読み込みました: ['counts_filtered.csv', 'vst.csv']


In [5]:
import pandas as pd

counts = pd.read_csv("counts_filtered.csv", index_col="gene")
samples = pd.read_csv("samples.csv", index_col="sample").drop(columns=["accession"])
samples = samples.loc[counts.columns]
print(counts.shape[0], "遺伝子 ×", counts.shape[1], "試料")
samples

13131 遺伝子 × 6 試料


,condition
shControl_1,shControl
shControl_2,shControl
shControl_3,shControl
shPPP3CA_1,shPPP3CA
shPPP3CA_2,shPPP3CA
shPPP3CA_3,shPPP3CA


## DESeq2 を実行する

`design="~condition"` は、「`condition` 列の違いで比べる」という指定です。

(実行時間: 約1分)

In [6]:
from pydeseq2.dds import DeseqDataSet

dds = DeseqDataSet(counts=counts.T, metadata=samples, design="~condition", quiet=True)
dds.deseq2()
print("終わりました")

終わりました


## サイズファクター

サイズファクターは、試料ごとの読み取った量の違いを表す係数です。カウントをサイズファクターで割ったものを、正規化したカウントと呼びます。

DESeq2 のサイズファクターは、合計で割るだけの計算ではありません。遺伝子ごとに「この試料の値 ÷ 全試料の平均的な値」の比を求め、その中央値を使います。こうすると、ごく一部の遺伝子のカウントが極端に多くても、その影響を受けにくくなります。

In [7]:
size_factors = pd.DataFrame({
    "size_factor": dds.obs["size_factors"],
    "total_counts": counts.sum(),
})
size_factors["total / mean(total)"] = size_factors["total_counts"] / size_factors["total_counts"].mean()
size_factors.round(3)

,size_factor,total_counts,total / mean(total)
shControl_1,1.101,18810876,1.099
shControl_2,1.014,17213599,1.006
shControl_3,0.908,15372520,0.898
shPPP3CA_1,1.050,17837069,1.042
shPPP3CA_2,0.940,15963055,0.932
shPPP3CA_3,1.020,17514855,1.023


サイズファクターと、合計を平均で割った値を見比べてください。

## 正規化したカウント

*PPP3CA* について、元のカウントと、正規化したカウントを並べます。

In [8]:
normalized = pd.DataFrame(dds.layers["normed_counts"].T, index=counts.index, columns=counts.columns)

GENE = "PPP3CA"
pd.DataFrame({"raw": counts.loc[GENE], "normalized": normalized.loc[GENE].round(1)})

,raw,normalized
shControl_1,4343,3943.0
shControl_2,4230,4169.8
shControl_3,3808,4195.0
shPPP3CA_1,631,601.1
shPPP3CA_2,579,616.1
shPPP3CA_3,549,538.0


## shPPP3CA と shControl を比べる

`contrast=["condition", "shPPP3CA", "shControl"]` で、「shPPP3CA ÷ shControl」の向きで比べます。log₂ fold change が正なら shPPP3CA で増え、負なら減っています。

In [9]:
from pydeseq2.ds import DeseqStats

stats = DeseqStats(dds, contrast=["condition", "shPPP3CA", "shControl"], quiet=True)
stats.summary()
res = stats.results_df.sort_values("padj")
res.head(10)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
gene,,,,,,
PPP3CA,2343.823799,-2.810328,0.071559,-39.273027,0.000000e+00,0.000000e+00
ABCC3,3120.805839,1.313285,0.057485,22.845774,1.609841e-115,9.749198e-112
C3,3026.870172,1.312424,0.058339,22.496474,4.494035e-112,1.814392e-108
MRPL9,1292.949607,-1.670468,0.075624,-22.089161,4.017866e-108,1.216610e-104
SAT1,2418.430792,1.319066,0.068347,19.299432,5.430297e-83,1.315435e-79
CFB,1916.354166,1.249761,0.065366,19.119313,1.743890e-81,3.520332e-78
KRT80,2336.632646,1.151705,0.060373,19.076604,3.951086e-81,6.836508e-78
NRP1,1696.789278,1.262609,0.067349,18.747246,2.038809e-78,2.743784e-75
SCD,12640.454860,-0.912885,0.048680,-18.752658,1.841537e-78,2.743784e-75


結果の表の列の意味は次のとおりです。

| 列 | 意味 |
|---|---|
| `baseMean` | 6試料の正規化したカウントの平均 |
| `log2FoldChange` | 変化の大きさ。2倍なら +1、同じなら 0、半分なら −1 |
| `lfcSE` | log2FoldChange の見積もりの不確かさ（標準誤差） |
| `stat` | log2FoldChange ÷ lfcSE。検定に使う値 |
| `pvalue` | 本当は差がないとしたときに、これくらい以上の差が偶然に出る確率 |
| `padj` | たくさんの遺伝子を同時に調べたことを考えて調整した p値（Benjamini–Hochberg 法） |

`padj` が `NaN`（空欄）の遺伝子は、外れ値があるなどの理由で、DESeq2 が検定から外したものです。

## 変化した遺伝子を数える

ここでは「padj < 0.05」を、変化があったとみなす基準にします。さらに「変化の大きさが2倍以上（|log₂FC| ≥ 1）」も加えた数も数えます。どちらもこの実習で決めた基準です。

In [10]:
PADJ = 0.05
LFC = 1

sig = res[res["padj"] < PADJ]
print(f"padj < {PADJ}:", len(sig), "遺伝子")
print("  そのうち shPPP3CA で増加:", (sig["log2FoldChange"] > 0).sum())
print("  そのうち shPPP3CA で減少:", (sig["log2FoldChange"] < 0).sum())
print(f"padj < {PADJ} かつ |log2FC| >= {LFC}:", (sig["log2FoldChange"].abs() >= LFC).sum(), "遺伝子")

padj < 0.05: 2559 遺伝子
  そのうち shPPP3CA で増加: 1287
  そのうち shPPP3CA で減少: 1272
padj < 0.05 かつ |log2FC| >= 1: 127 遺伝子


## 遺伝子を調べる

`GENES` に遺伝子名を加えると、その遺伝子の結果を並べて見られます。

In [11]:
GENES = ["PPP3CA", "E2F1", "GAPDH"]

res.loc[GENES]

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
gene,,,,,,
PPP3CA,2343.823799,-2.810328,0.071559,-39.273027,0.000000,0.000000
E2F1,1007.928645,-0.323780,0.112293,-2.883354,0.003935,0.022751
GAPDH,56957.895680,-0.106786,0.046312,-2.305779,0.021123,0.085086


## 確かめよう

1. *PPP3CA* の log2FoldChange はいくつですか。何分の1に減ったことになりますか（ヒント：2 の −log2FoldChange 乗）。
2. *E2F1* の RNA 量は、どれくらい変わっていますか。論文の「カルシニューリンが E2F1 のタンパク質を安定にする」という話と合わせて考えてみましょう。RNA-seq で測っているのは何でしたか。
3. `PADJ` を 0.01 に変えると、変化した遺伝子の数はどう変わりますか。

In [12]:
res.to_csv("deseq2_results.csv")
normalized.to_csv("normalized_counts.csv")

## 結果をダウンロードする

下のセルを実行すると、ここまでの結果をまとめた `results_03.zip` がパソコンにダウンロードされます。次の 04_Visualization の最初でアップロードするので、保存した場所を覚えておいてください。

In [13]:
import zipfile

OUT = "results_03.zip"
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ['counts_filtered.csv', 'vst.csv', 'deseq2_results.csv', 'normalized_counts.csv']:
        z.write(name)
print(OUT, "に入れたファイル:", zipfile.ZipFile(OUT).namelist())

try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    pass  # Colab 以外で実行したときは、手元のフォルダに保存される

results_03.zip に入れたファイル: ['counts_filtered.csv', 'vst.csv', 'deseq2_results.csv', 'normalized_counts.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

次は **04_Visualization** に進みます。

[https://colab.research.google.com/github/Yokohide0317/what_is_BI/blob/main/public/colab/04_Visualization.ipynb](https://colab.research.google.com/github/Yokohide0317/what_is_BI/blob/main/public/colab/04_Visualization.ipynb)